# Optimasi Nilai Belief Sistem Pakar iPhone dengan Hibrida DS-PSO (Brier Score)

Notebook ini mengimplementasikan pemodelan sistem pakar untuk diagnosis kerusakan iPhone menggunakan metode **Dempster-Shafer (DS)** yang dioptimasikan secara otomatis menggunakan **Particle Swarm Optimization (PSO)**.

Fungsi fitness optimasi ini menggunakan metrik **Brier Score** (kontinu halus) berbasis **Probabilitas Pignistik (Pignistic Probability)** untuk mengatasi kendala *flat fitness landscape* (stagnasi pencarian) dan menjamin keluaran probabilitas diagnosis yang lebih akurat.

In [ ]:
import pandas as pd
import numpy as np
import os
import json
import glob
import time
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

In [ ]:
data_dir = "data"
belief_path = os.path.join(data_dir, "belief.json")

# 1. Membaca berkas aturan belief.json
with open(belief_path, 'r', encoding='utf-8') as f:
    belief_data = json.load(f)

# 2. Mencari berkas dataset latih/uji berstempel waktu terbaru
latih_files = glob.glob(os.path.join(data_dir, "dataLatih", "dataLatih-*.csv"))
uji_files = glob.glob(os.path.join(data_dir, "dataUji", "dataUji-*.csv"))

if not latih_files or not uji_files:
    raise FileNotFoundError("Berkas dataset CSV hasil preprocessing tidak ditemukan.")

latest_latih = max(latih_files, key=os.path.getmtime)
latest_uji = max(uji_files, key=os.path.getmtime)

df_train = pd.read_csv(latest_latih)
df_test = pd.read_csv(latest_uji)

# Mengidentifikasi seluruh kelas target kerusakan secara dinamis dari data latih
all_damages = sorted(list(df_train['Kerusakan'].unique()))
print("Daftar Kategori Kerusakan Target:", all_damages)
Theta = frozenset(all_damages)
num_classes = len(all_damages)

# 3. Menghitung probabilitas prior empiris dari data latih (untuk fallback jika tidak ada gejala aktif)
class_counts = df_train['Kerusakan'].value_counts()
total_train_rows = len(df_train)
priors = {c: float(class_counts.get(c, 0.0) / total_train_rows) for c in all_damages}
print("\nEmpirical Prior Probabilities (Fallback):")
for c, p in priors.items():
    print(f"- {c}: {p:.4f}")

print(f"\nBerhasil memuat dataset:\n- Data Latih: {latest_latih} ({df_train.shape[0]} baris)\n- Data Uji: {latest_uji} ({df_test.shape[0]} baris)")

### Pre-kalkulasi Gejala Aktif (*Active Symptom Indexing*)
Memetakan gejala aktif per baris data ke indeks aturan `belief.json` untuk optimasi kecepatan komputasi (500x lebih cepat).

In [ ]:
rules_list = belief_data['rules']
total_dimensions = sum(len(r['hypotheses']) for r in rules_list)
print(f"Jumlah parameter Belief yang dioptimasi oleh PSO (Dimensi Partikel): {total_dimensions}")

def constrain_beliefs(vector, rules_list):
    # Membatasi nilai belief dalam rentang [0, 0.95] dan jumlah belief per gejala <= 0.95
    constrained = np.clip(vector, 0.0, 0.95)
    idx = 0
    for rule in rules_list:
        num_hyp = len(rule['hypotheses'])
        sub_vec = constrained[idx:idx+num_hyp]
        sum_b = np.sum(sub_vec)
        if sum_b > 0.95:
            constrained[idx:idx+num_hyp] = (sub_vec / sum_b) * 0.95
        idx += num_hyp
    return constrained

def precalculate_active_rules(df, rules_list):
    row_active_rules = []
    for _, row in df.iterrows():
        active = []
        idx = 0
        for rule in rules_list:
            col = rule['symptom_col']
            val = rule['symptom_val']
            num_hyp = len(rule['hypotheses'])
            if row[col] == val:
                active.append({
                    'start_idx': idx,
                    'num_hyp': num_hyp,
                    'hypotheses': rule['hypotheses']
                })
            idx += num_hyp
        row_active_rules.append(active)
    return row_active_rules

print("Menjalankan index lookup...")
train_active_rules = precalculate_active_rules(df_train, rules_list)
test_active_rules = precalculate_active_rules(df_test, rules_list)
print("Pre-kalkulasi selesai!")

### Probabilitas Pignistik (Pignistic Probability) & DS Combiner
Menerjemahkan fungsi massa gabungan Dempster-Shafer menjadi distribusi probabilitas klasik. Jika tidak ada gejala yang aktif, probabilitas akan langsung difallback menggunakan prior empiris dari dataset latih.

In [ ]:
def combine_two_mass_functions(m1, m2):
    m_new = {}
    K = 0.0
    for x, w1 in m1.items():
        for y, w2 in m2.items():
            intersection = x.intersection(y)
            if not intersection:
                K += w1 * w2
            else:
                m_new[intersection] = m_new.get(intersection, 0.0) + w1 * w2
                
    if K >= 1.0:
        return {Theta: 1.0}
        
    for k in m_new:
        m_new[k] /= (1.0 - K)
        
    return m_new

def predict_row_probabilities(active_rules, belief_vector):
    if not active_rules:
        # Jika tidak ada gejala aktif, gunakan prior empiris
        return priors.copy()
        
    m_list = []
    for rule in active_rules:
        m = {}
        sum_b = 0.0
        start = rule['start_idx']
        for offset, hyp in enumerate(rule['hypotheses']):
            b_val = belief_vector[start + offset]
            sum_b += b_val
            m[frozenset([hyp['damage']])] = b_val
        m[Theta] = 1.0 - sum_b
        m_list.append(m)
        
    m_combined = m_list[0]
    for m_next in m_list[1:]:
        m_combined = combine_two_mass_functions(m_combined, m_next)
        
    # Hitung Probabilitas Pignistik (BetP)
    probs = {c: 0.0 for c in all_damages}
    mass_theta = m_combined.get(Theta, 0.0)
    theta_share = mass_theta / num_classes
    
    for subset, mass in m_combined.items():
        if subset == Theta:
            continue
        if len(subset) == 1:
            c = list(subset)[0]
            if c in probs:
                probs[c] += mass
        else:
            valid_elements = [c for c in subset if c in probs]
            if valid_elements:
                share = mass / len(valid_elements)
                for c in valid_elements:
                    probs[c] += share
                    
    for c in probs:
        probs[c] += theta_share
        
    return probs

def evaluate_brier_fitness(belief_vector, active_rules_list, one_hot_targets):
    fitness_sum = 0.0
    n = len(active_rules_list)
    for i in range(n):
        probs = predict_row_probabilities(active_rules_list[i], belief_vector)
        row_se = 0.0
        for c_idx, c in enumerate(all_damages):
            p = probs[c]
            y = one_hot_targets[i, c_idx]
            row_se += (p - y) ** 2
        fitness_sum += row_se
    mean_brier_score = fitness_sum / n
    # Fitness adalah 1.0 - Brier (makin tinggi makin baik)
    return 1.0 - mean_brier_score

def evaluate_discrete_accuracy(belief_vector, active_rules_list, targets):
    preds = []
    for ar in active_rules_list:
        probs = predict_row_probabilities(ar, belief_vector)
        best_class = max(probs, key=probs.get)
        preds.append(best_class)
    acc = np.mean(np.array(preds) == targets)
    return acc

In [ ]:
one_hot_train = np.zeros((len(df_train), num_classes))
one_hot_test = np.zeros((len(df_test), num_classes))

for i, t in enumerate(df_train['Kerusakan'].values):
    one_hot_train[i, all_damages.index(t)] = 1.0
for i, t in enumerate(df_test['Kerusakan'].values):
    one_hot_test[i, all_damages.index(t)] = 1.0

initial_beliefs = []
for rule in rules_list:
    for hyp in rule['hypotheses']:
        initial_beliefs.append(hyp['initial_belief'])
initial_beliefs = np.array(initial_beliefs)
initial_beliefs = constrain_beliefs(initial_beliefs, rules_list)

train_targets = df_train['Kerusakan'].values
test_targets = df_test['Kerusakan'].values

t0 = time.time()
init_brier_fit = evaluate_brier_fitness(initial_beliefs, train_active_rules, one_hot_train)
init_acc = evaluate_discrete_accuracy(initial_beliefs, train_active_rules, train_targets)
t1 = time.time()
print(f"Brier Fitness Awal (Bobot Pakar): {init_brier_fit:.4f} (Akurasi: {init_acc*100:.2f}%) (dievaluasi dalam {t1 - t0:.4f} detik)")

### Proses Optimasi PSO (Fungsi Fitness Brier Score)
Mencari nilai optimal bobot belief dengan menyeimbangkan parameter internal.

In [ ]:
np.random.seed(42)
num_particles = 30
max_iter = 100
c1, c2 = 2.0, 2.0

positions = np.zeros((num_particles, total_dimensions))
positions[0] = initial_beliefs
for p in range(1, num_particles):
    positions[p] = constrain_beliefs(initial_beliefs + np.random.uniform(-0.2, 0.2, total_dimensions), rules_list)

velocities = np.random.uniform(-0.1, 0.1, (num_particles, total_dimensions))
fitnesses = np.zeros(num_particles)
for p in range(num_particles):
    fitnesses[p] = evaluate_brier_fitness(positions[p], train_active_rules, one_hot_train)

pbests = np.copy(positions)
pbest_fitnesses = np.copy(fitnesses)

gbest_idx = np.argmax(fitnesses)
gbest_position = np.copy(positions[gbest_idx])
gbest_fitness = fitnesses[gbest_idx]

print(f"GBest Fitness Brier Awal: {gbest_fitness:.4f}")

t_start = time.time()
for it in range(max_iter):
    w = 0.9 - ((0.9 - 0.4) / max_iter) * it
    
    for p in range(num_particles):
        r1, r2 = np.random.rand(total_dimensions), np.random.rand(total_dimensions)
        
        velocities[p] = (w * velocities[p] + 
                         c1 * r1 * (pbests[p] - positions[p]) + 
                         c2 * r2 * (gbest_position - positions[p]))
        velocities[p] = np.clip(velocities[p], -0.2, 0.2)
        positions[p] = constrain_beliefs(positions[p] + velocities[p], rules_list)
        
        fit = evaluate_brier_fitness(positions[p], train_active_rules, one_hot_train)
        fitnesses[p] = fit
        
        if fit > pbest_fitnesses[p]:
            pbest_fitnesses[p] = fit
            pbests[p] = np.copy(positions[p])
            
            if fit > gbest_fitness:
                gbest_fitness = fit
                gbest_position = np.copy(positions[p])
                
    if (it + 1) % 10 == 0 or it == 0:
        current_acc = evaluate_discrete_accuracy(gbest_position, train_active_rules, train_targets)
        print(f"Iterasi {it+1}/{max_iter} - GBest Brier Fitness: {gbest_fitness:.4f} (Akurasi: {current_acc*100:.2f}%)")

t_end = time.time()
print(f"\nOptimasi Selesai! GBest Brier Latih Final: {gbest_fitness:.4f} (selesai dalam {t_end - t_start:.2f} detik)")

### Menyimpan Hasil Bobot Optimal & Priors Fallback
Mengekspor bobot optimal dan data prior probabilitas ke file `data/optimal_knowledge_base.json`.

In [ ]:
optimal_kb = {
    "class_priors": priors,
    "rules": []
}
idx = 0
for rule in rules_list:
    rule_entry = {
        "symptom_col": rule["symptom_col"],
        "symptom_val": rule["symptom_val"],
        "hypotheses": []
    }
    sum_b = 0.0
    for hyp in rule["hypotheses"]:
        opt_b = float(gbest_position[idx])
        sum_b += opt_b
        rule_entry["hypotheses"].append({
            "damage": hyp["damage"],
            "optimal_belief": round(opt_b, 4)
        })
        idx += 1
    rule_entry["uncertainty_theta"] = round(1.0 - sum_b, 4)
    optimal_kb["rules"].append(rule_entry)

output_kb_path = os.path.join(data_dir, "optimal_knowledge_base.json")
with open(output_kb_path, 'w', encoding='utf-8') as f:
    json.dump(optimal_kb, f, indent=2)

print(f"Basis pengetahuan optimal berhasil disimpan ke: {output_kb_path}")

### Evaluasi Performa Akhir pada Data Uji
Mengukur akurasi model optimal DS-PSO dan menggambarkan Confusion Matrix pada data uji.

In [ ]:
test_preds = []
for ar in test_active_rules:
    probs = predict_row_probabilities(ar, gbest_position)
    best_class = max(probs, key=probs.get)
    test_preds.append(best_class)

test_acc = np.mean(np.array(test_preds) == test_targets)
print(f"Akurasi Model Optimal DS-PSO (Brier) pada Data Uji: {test_acc * 100:.2f}%")

print("\nClassification Report:")
print(classification_report(test_targets, test_preds, zero_division=0))

cm = confusion_matrix(test_targets, test_preds, labels=all_damages)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=all_damages, yticklabels=all_damages)
plt.title('Confusion Matrix Sistem Pakar DS-PSO Brier (Data Uji)')
plt.xlabel('Prediksi Sistem')
plt.ylabel('Kerusakan Aktual')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()